# Final Task B 17-class - ResNet-50

Train ResNet-50 directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-old"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-old: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
final-B: /workspace/woodbench/notebooks/phase 1/final-B


In [2]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [3]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "resnet50"
DISPLAY_NAME = "ResNet-50"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: ResNet-50 (resnet50)
Batch size: 64
Data root: /workspace/woodbench/data/task-B-group-split


In [4]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-17class

main/seed_42/resnet50
[main | resnet50 s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8415, Acc: 5.02% | Val Loss: 2.8499, Acc: 3.88% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 3.88%)


Epoch 2/50 | Train Loss: 2.8311, Acc: 6.79% | Val Loss: 2.8303, Acc: 5.98% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 5.98%)


Epoch 3/50 | Train Loss: 2.8046, Acc: 12.23% | Val Loss: 2.7958, Acc: 8.42% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.42%)


Epoch 4/50 | Train Loss: 2.7626, Acc: 20.55% | Val Loss: 2.7461, Acc: 18.49% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 18.49%)


... (92 outputs left out)

Epoch 47/50 | Train Loss: 0.8197, Acc: 94.14% | Val Loss: 0.9154, Acc: 89.81% | LR: 1.09e-06[EarlyStop 2/10]


Epoch 48/50 | Train Loss: 0.8124, Acc: 94.11% | Val Loss: 0.9084, Acc: 89.81% | LR: 4.87e-07[EarlyStop 3/10]


Epoch 49/50 | Train Loss: 0.8231, Acc: 93.90% | Val Loss: 0.9132, Acc: 90.37% | LR: 1.22e-07[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed123_best.pth (Val Acc: 90.37%)


Epoch 50/50 [Train]:  46%|████▌     | 24/52 [01:55<01:29,  3.19s/batch, loss=0.7752, acc=94.0%]

In [ ]:
summarize_results(results)


## Continue after Vast interruption

Run the setup/data-check cells above, then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [8]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [9]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 2/6 completed configs for resnet50.


In [10]:
# Remaining main run (not previously started)
run_remaining("main", 456)


[main | resnet50 s456] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8391, Acc: 6.58% | Val Loss: 2.8378, Acc: 6.09% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed456_best.pth (Val Acc: 6.09%)


Epoch 2/50 | Train Loss: 2.8301, Acc: 8.38% | Val Loss: 2.8184, Acc: 9.30% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed456_best.pth (Val Acc: 9.30%)


Epoch 3/50 | Train Loss: 2.8019, Acc: 15.05% | Val Loss: 2.7816, Acc: 20.60% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed456_best.pth (Val Acc: 20.60%)


Epoch 4/50 | Train Loss: 2.7587, Acc: 21.63% | Val Loss: 2.7344, Acc: 28.46% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed456_best.pth (Val Acc: 28.46%)


Epoch 5/50 | Train Loss: 2.6964, Acc: 32.51% | Val Loss: 2.6626, Acc: 33.67% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed456_best.pth (Val Acc: 33.67%)


... (43 outputs left out)

Epoch 49/50 | Train Loss: 0.8101, Acc: 94.08% | Val Loss: 0.8897, Acc: 90.37% | LR: 1.22e-07[EarlyStop 3/10]


Epoch 50/50 | Train Loss: 0.8067, Acc: 94.32% | Val Loss: 0.8876, Acc: 90.25% | LR: 0.00e+00[EarlyStop 4/10]


[main | resnet50 s456] val_acc=90.59% test_acc=93.61% F1=93.75%


{'backbone': 'resnet50',
 'seed': 456,
 'experiment': 'main',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 90.58693244739756,
 'best_epoch': 46,
 'final_train_acc': 94.32091346153847,
 'final_val_acc': 90.2547065337763,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9360902255639098,
 'test_f1_macro': 0.9374935736438277,
 'test_precision_macro': 0.9454711061328709,
 'test_recall_macro': 0.9373779538251019,
 'test_f1_per_class': {'Araucaria araucana': 0.9285714285714286,
  'Callitris columellaris': 0.9393939393939394,
  'Calocedrus rupestris': 1.0,
  'Ch

In [11]:
# Remaining 90-degree rotation ablation
run_remaining("taskB_90rotation", 42)


[taskB_90rotation | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8378, Acc: 5.14% | Val Loss: 2.8420, Acc: 4.21% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 4.21%)


Epoch 2/50 | Train Loss: 2.8257, Acc: 7.09% | Val Loss: 2.8137, Acc: 9.75% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 9.75%)


Epoch 3/50 | Train Loss: 2.7911, Acc: 14.48% | Val Loss: 2.7602, Acc: 23.03% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 23.03%)


Epoch 4/50 | Train Loss: 2.7348, Acc: 26.62% | Val Loss: 2.6814, Acc: 34.99% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 34.99%)


Epoch 5/50 | Train Loss: 2.6563, Acc: 38.61% | Val Loss: 2.5804, Acc: 47.51% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 47.51%)


... (42 outputs left out)

Epoch 48/50 | Train Loss: 0.8255, Acc: 93.63% | Val Loss: 0.8270, Acc: 92.58% | LR: 4.87e-07[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 92.58%)


Epoch 49/50 | Train Loss: 0.8238, Acc: 93.60% | Val Loss: 0.8251, Acc: 92.36% | LR: 1.22e-07[EarlyStop 1/10]


Epoch 50/50 | Train Loss: 0.8165, Acc: 94.38% | Val Loss: 0.8289, Acc: 92.14% | LR: 0.00e+00[EarlyStop 2/10]
[taskB_90rotation | resnet50 s42] val_acc=92.58% test_acc=95.68% F1=95.73%


{'backbone': 'resnet50',
 'seed': 42,
 'experiment': 'taskB_90rotation',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 92.58028792912513,
 'best_epoch': 48,
 'final_train_acc': 94.38100961538461,
 'final_val_acc': 92.13732004429679,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.956766917293233,
 'test_f1_macro': 0.9572853750137978,
 'test_precision_macro': 0.9622592870165143,
 'test_recall_macro': 0.9563746538122864,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9850746268656716,
  'Calocedrus rupestris': 1.0,
  'Chamae

In [12]:
# Remaining label-smoothing=0.0 ablation
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8422, Acc: 5.02% | Val Loss: 2.8516, Acc: 3.88% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 3.88%)


Epoch 2/50 | Train Loss: 2.8306, Acc: 6.79% | Val Loss: 2.8298, Acc: 5.98% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 5.98%)


Epoch 3/50 | Train Loss: 2.8010, Acc: 11.87% | Val Loss: 2.7910, Acc: 8.64% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.64%)


Epoch 4/50 | Train Loss: 2.7536, Acc: 20.16% | Val Loss: 2.7343, Acc: 17.61% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 17.61%)


Epoch 5/50 | Train Loss: 2.6810, Acc: 29.30% | Val Loss: 2.6472, Acc: 29.01% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 29.01%)


... (42 outputs left out)

Epoch 48/50 | Train Loss: 0.2713, Acc: 93.06% | Val Loss: 0.3713, Acc: 89.15% | LR: 4.87e-07[EarlyStop 2/10]


Epoch 49/50 | Train Loss: 0.2713, Acc: 93.72% | Val Loss: 0.3807, Acc: 89.81% | LR: 1.22e-07[EarlyStop 3/10]


Epoch 50/50 | Train Loss: 0.2704, Acc: 93.69% | Val Loss: 0.3636, Acc: 90.59% | LR: 0.00e+00[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 90.59%)
[taskB_label_smoothing_0.0 | resnet50 s42] val_acc=90.59% test_acc=93.80% F1=93.95%


{'backbone': 'resnet50',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 90.58693244739756,
 'best_epoch': 50,
 'final_train_acc': 93.68990384615384,
 'final_val_acc': 90.58693244739756,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9379699248120301,
 'test_f1_macro': 0.9395075179497383,
 'test_precision_macro': 0.9455418932278794,
 'test_recall_macro': 0.9391909948466205,
 'test_f1_per_class': {'Araucaria araucana': 0.9830508474576272,
  'Callitris columellaris': 0.927536231884058,
  'Calocedrus ru

In [13]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8409, Acc: 5.02% | Val Loss: 2.8483, Acc: 3.88% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 3.88%)


Epoch 2/50 | Train Loss: 2.8316, Acc: 6.76% | Val Loss: 2.8309, Acc: 5.98% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 5.98%)


Epoch 3/50 | Train Loss: 2.8082, Acc: 12.20% | Val Loss: 2.8007, Acc: 8.19% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.19%)


Epoch 4/50 | Train Loss: 2.7715, Acc: 20.94% | Val Loss: 2.7577, Acc: 18.83% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 18.83%)


Epoch 5/50 | Train Loss: 2.7154, Acc: 30.26% | Val Loss: 2.6917, Acc: 31.56% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 31.56%)


... (43 outputs left out)

Epoch 49/50 | Train Loss: 1.2225, Acc: 94.38% | Val Loss: 1.2974, Acc: 89.92% | LR: 1.22e-07[EarlyStop 7/10]


Epoch 50/50 | Train Loss: 1.2208, Acc: 94.29% | Val Loss: 1.2888, Acc: 90.59% | LR: 0.00e+00[EarlyStop 8/10]


[taskB_label_smoothing_0.2 | resnet50 s42] val_acc=90.70% test_acc=94.17% F1=94.29%


{'backbone': 'resnet50',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 90.69767441860465,
 'best_epoch': 42,
 'final_train_acc': 94.29086538461539,
 'final_val_acc': 90.58693244739756,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9417293233082706,
 'test_f1_macro': 0.942851224921522,
 'test_precision_macro': 0.9487724501565333,
 'test_recall_macro': 0.942186200653245,
 'test_f1_per_class': {'Araucaria araucana': 0.9473684210526315,
  'Callitris columellaris': 0.9565217391304348,
  'Calocedrus rup

In [14]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    90.59%    94.55%    94.69%    95.16%    94.65%
main                              123    90.37%    92.29%    92.34%    93.40%    92.43%
main                              456    90.59%    93.61%    93.75%    94.55%    93.74%
taskB_90rotation                   42    92.58%    95.68%    95.73%    96.23%    95.64%
taskB_label_smoothing_0.0          42    90.59%    93.80%    93.95%    94.55%    93.92%
taskB_label_smoothing_0.2          42    90.70%    94.17%    94.29%    94.88%    94.22%
----------------------------------------------------------------------------------------
main test_acc: 93.48 +/- 0.93%
main test_f1:  93.59 +/- 0.97%
